# Enriquecimento de Features V2

A primeira rodada de modelagem apresentou baixo poder discriminatório para a classificação individual dos alunos.

O diagnóstico posterior indicou que a base estava estruturalmente consistente, porém as features utilizadas eram predominantemente contextuais e apresentavam capacidade limitada de separação entre alunos alfabetizados e não alfabetizados.

Esta etapa inicia uma segunda iteração de engenharia de features, priorizando informações educacionais mais diretamente relacionadas ao processo de aprendizagem.

As novas fontes serão avaliadas antes de sua incorporação à base analítica, considerando:

- relevância conceitual;
- disponibilidade temporal;
- granularidade;
- cobertura da população de modelagem;
- risco de vazamento de informação;
- contribuição de uma dimensão de informação ainda não representada na base.

A primeira fonte avaliada será o histórico do IDEB.

# 1. Configuração
Nesta etapa são importadas as bibliotecas utilizadas no notebook

## Bibliotecas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

from pyspark.ml.evaluation import BinaryClassificationEvaluator

## Conexão Base-Dados

In [0]:
credencial_gcp = dbutils.secrets.get(
    scope="fiap-tech-challenge",
    key="gcp-service-account-b64"
)

if not credencial_gcp:
    raise ValueError("Credencial do Google Cloud não encontrada.")

print("Credencial carregada com sucesso.")

In [0]:
dbutils.widgets.text(
    "billing_project_id",
    "fiap-tech-challenge2",
    "Billing Project GCP"
)

billing_project_id = dbutils.widgets.get("billing_project_id")

print(f"Billing project: {billing_project_id}")

# 2. Avaliação do IDEB Histórico

O IDEB combina informações de desempenho educacional e fluxo escolar, podendo representar uma dimensão diferente das características socioeconômicas e de infraestrutura utilizadas na primeira versão da base.

A utilização dessas informações deve respeitar a temporalidade da modelagem. Indicadores produzidos no mesmo período ou posteriormente ao target não poderão ser utilizados como features.

Antes da seleção das variáveis, são avaliados o schema, a cobertura temporal e as categorias disponíveis na tabela municipal.

## 2.1 Carregamento

In [0]:
df_ideb = (
    spark.read
    .format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load("basedosdados.br_inep_ideb.municipio")
)

## 2.2 Discover

Schema

In [0]:
df_ideb.printSchema()

Temporalidade

In [0]:
display(df_ideb.groupBy("ano").count().orderBy(F.desc("ano")))

Variáveis

In [0]:
display(
    df_ideb.filter(F.col("ano").isin(2021, 2023))
    .groupBy("ano", "rede", "ensino", "anos_escolares")
    .count()
    .orderBy("ano", "rede", "ensino", "anos_escolares")
)

Preenchimento das variáveis

In [0]:
metricas_ideb = [
    "taxa_aprovacao",
    "indicador_rendimento",
    "nota_saeb_matematica",
    "nota_saeb_lingua_portuguesa",
    "nota_saeb_media_padronizada",
    "ideb"
]

expressoes = [
    F.count("*").alias("qtd_registros")
]

for coluna in metricas_ideb:
    expressoes.append(
        F.count(coluna).alias(f"preenchidos_{coluna}")
    )

display(
    df_ideb
    .filter(F.col("ano").isin(2021, 2023))
    .groupBy("ano")
    .agg(*expressoes)
    .orderBy("ano")
)

Cobertura

In [0]:
df_ideb_cobertura = (
    df_ideb.filter(F.col("ano") == 2021)
    .filter(F.col("rede").isin("estadual", "municipal"))
    .filter(F.col("ensino") == "fundamental")
    .filter(F.col("anos_escolares") == "iniciais (1-5)")
    .withColumn(
        "rede_codigo",
        F.when(F.col("rede") == "estadual", F.lit("2")).when(
            F.col("rede") == "municipal", F.lit("3")
        ),
    )
)

df_base_atual = spark.table("workspace.gold.ml_alfabetizacao_aluno")

df_cobertura_ideb = (
    df_base_atual.alias("base")
    .join(
        df_ideb_cobertura.alias("ideb"),
        (F.col("base.id_municipio") == F.col("ideb.id_municipio"))
        & (F.col("base.rede") == F.col("ideb.rede_codigo")),
        how="left",
    )
    .select(
        "base.ano",
        "base.rede",
        F.col("ideb.taxa_aprovacao").alias("taxa_aprovacao"),
        F.col("ideb.indicador_rendimento").alias("indicador_rendimento"),
        F.col("ideb.nota_saeb_media_padronizada").alias("nota_saeb_media"),
        F.col("ideb.ideb").alias("ideb"),
    )
)

display(
    df_cobertura_ideb.groupBy("ano", "rede")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.count("taxa_aprovacao").alias("com_taxa_aprovacao"),
        F.count("indicador_rendimento").alias("com_indicador_rendimento"),
        F.count("nota_saeb_media").alias("com_saeb"),
        F.count("ideb").alias("com_ideb"),
    )
    .withColumn(
        "pct_taxa_aprovacao",
        F.round(F.col("com_taxa_aprovacao") / F.col("qtd_alunos") * 100, 2),
    )
    .withColumn(
        "pct_indicador_rendimento",
        F.round(F.col("com_indicador_rendimento") / F.col("qtd_alunos") * 100, 2),
    )
    .withColumn("pct_saeb", F.round(F.col("com_saeb") / F.col("qtd_alunos") * 100, 2))
    .withColumn("pct_ideb", F.round(F.col("com_ideb") / F.col("qtd_alunos") * 100, 2))
    .orderBy("ano", "rede")
)

## 2.3 Teste de Aderência

Para evitar qualquer ambiguidade relacionada à disponibilidade das informações, são utilizadas apenas edições do IDEB anteriores ao período de modelagem.

A edição de 2021 permite representar o nível recente de desempenho educacional

O recorte considera exclusivamente:

- Ensino Fundamental;
- anos iniciais (1º ao 5º ano);
- redes estadual e municipal.

A categoria de rede pública agregada não é utilizada, pois a base de alfabetização diferencia as redes estadual e municipal.

### Carregando IDEB

In [0]:
df_ideb_selecionado = (
    df_ideb.filter(F.col("ano").isin(2021))
    .filter(F.col("rede").isin("estadual", "municipal"))
    .filter(F.col("ensino") == "fundamental")
    .filter(F.col("anos_escolares") == "iniciais (1-5)")
    .withColumn(
        "rede_codigo",
        F.when(F.col("rede") == "estadual", F.lit("2")).when(
            F.col("rede") == "municipal", F.lit("3")
        ),
    )
)

### Cruzamento Base

Carregando Split

In [0]:
df_split = spark.table("workspace.gold.ml_split_modelagem")

df_desenvolvimento = df_base_atual.join(
    df_split, on=["ano", "id_municipio"], how="inner"
).filter(F.col("split").isin("treino", "validacao"))

Cruzando IDEB e Split

In [0]:
df_dev_ideb = (
    df_desenvolvimento.alias("base")
    .join(
        df_ideb_selecionado.alias("ideb"),
        (F.col("base.id_municipio") == F.col("ideb.id_municipio"))
        & (F.col("base.rede") == F.col("ideb.rede_codigo")),
        how="left",
    )
    .select(
        "base.*",
        F.col("ideb.taxa_aprovacao").alias("ideb_taxa_aprovacao_2021"),
        F.col("ideb.indicador_rendimento").alias("ideb_indicador_rendimento_2021"),
        F.col("ideb.nota_saeb_matematica").alias("ideb_saeb_matematica_2021"),
        F.col("ideb.nota_saeb_lingua_portuguesa").alias("ideb_saeb_portugues_2021"),
        F.col("ideb.nota_saeb_media_padronizada").alias("ideb_saeb_media_2021"),
        F.col("ideb.ideb").alias("ideb_2021"),
    )
)

### Poder Discriminatório

As métricas são avaliadas individualmente por meio da correlação com o target e do ROC-AUC univariado.

Essa análise permite verificar se o histórico de desempenho educacional apresenta sinal superior às variáveis contextuais utilizadas na primeira versão da base.

In [0]:
features_ideb = [
    "ideb_taxa_aprovacao_2021",
    "ideb_indicador_rendimento_2021",
    "ideb_saeb_matematica_2021",
    "ideb_saeb_portugues_2021",
    "ideb_saeb_media_2021",
    "ideb_2021"
]

Função

In [0]:
def medir_associacao_univariada(df, features, split_nome):
    resultados = []

    evaluator = BinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="score", metricName="areaUnderROC"
    )

    df_base = df.withColumn("label", F.col("alfabetizado").cast("double"))

    for feature in features:
        df_feature = df_base.filter(F.col(feature).isNotNull()).select(
            "label", F.col(feature).cast("double").alias("score")
        )

        correlacao = df_feature.agg(F.corr("score", "label").alias("corr")).first()[
            "corr"
        ]

        auc = evaluator.evaluate(df_feature)

        auc_discriminatorio = max(auc, 1 - auc)

        resultados.append((split_nome, feature, correlacao, auc, auc_discriminatorio))

    return resultados

Execução

In [0]:
resultados_ideb = []

for split_nome in ["treino", "validacao"]:
    resultados_ideb += medir_associacao_univariada(
        df_dev_ideb.filter(F.col("split") == split_nome), features_ideb, split_nome
    )

df_associacao_ideb = spark.createDataFrame(
    resultados_ideb,
    ["split", "feature", "correlacao", "roc_auc", "roc_auc_discriminatorio"],
)

display(df_associacao_ideb.orderBy("split", F.desc("roc_auc_discriminatorio")))

### Redundância métricas 

In [0]:
correlacoes_ideb = []

df_treino_ideb = df_dev_ideb.filter(F.col("split") == "treino")

for i, feature_1 in enumerate(features_ideb):
    for feature_2 in features_ideb[i + 1 :]:
        correlacao = df_treino_ideb.agg(
            F.corr(feature_1, feature_2).alias("correlacao")
        ).first()["correlacao"]

        correlacoes_ideb.append(
            (
                feature_1,
                feature_2,
                correlacao,
                abs(correlacao) if correlacao is not None else None,
            )
        )

In [0]:
df_correlacoes_ideb = spark.createDataFrame(
    correlacoes_ideb,
    [
        "feature_1",
        "feature_2",
        "correlacao",
        "correlacao_abs"
    ]
)

display(
    df_correlacoes_ideb
    .orderBy(
        F.desc("correlacao_abs")
    )
)

## 2.4 Conclusão Features do IDEB

As métricas históricas do IDEB apresentaram poder discriminatório superior às features utilizadas na primeira versão da base.

Na validação, o IDEB 2021 apresentou ROC-AUC univariado de aproximadamente 0,609, enquanto as métricas de desempenho do SAEB ficaram próximas de 0,605.
Esses valores são superiores aos observados para as features contextuais da primeira rodada de modelagem.

Entretanto, foi identificada elevada redundância entre algumas métricas.

Para evitar a inclusão de informações praticamente duplicadas, foram selecionadas duas dimensões complementares:

- nota SAEB de Língua Portuguesa de 2021, representando desempenho educacional
  histórico;
- indicador de rendimento de 2021, representando fluxo escolar.

O IDEB agregado não é incluído simultaneamente, pois combina dimensões já representadas pelas features selecionadas.

## 2.5 Dataframe Final IDEB

In [0]:
df_ideb_features = df_ideb_selecionado.select(
    "id_municipio",
    "rede_codigo",
    F.col("nota_saeb_lingua_portuguesa").alias("ideb_saeb_portugues_2021"),
    F.col("indicador_rendimento").alias("ideb_indicador_rendimento_2021"),
)

In [0]:
display(
    df_ideb_features.filter(
        F.col("ideb_saeb_portugues_2021").isNotNull()
        & F.col("ideb_indicador_rendimento_2021").isNotNull()
    ).limit(10)
)

# 3. Avaliação do SAEB Histórico

Após a seleção das métricas do IDEB, é avaliado o SAEB histórico como segunda fonte de enriquecimento.

A análise prioriza informações relacionadas ao 2º ano do Ensino Fundamental, por apresentarem maior proximidade conceitual com o target de alfabetização.

Somente indicadores que acrescentem uma dimensão informacional relevante serão incorporados à base analítica V2.

## 3.1 Carregamento

In [0]:
df_saeb = (
    spark.read.format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load("basedosdados.br_inep_saeb.proficiencia")
)

## 3.2 Discover

Schema

In [0]:
df_saeb.printSchema()

Temporalidade

In [0]:
display(df_saeb.groupBy("ano").count().orderBy(F.desc("ano")))

Variáveis

In [0]:
display(
    df_saeb.filter(F.col("ano").isin(2019, 2021))
    .groupBy("ano", "serie", "disciplina")
    .count()
    .orderBy("ano", "serie", "disciplina")
)

In [0]:
display(
    df_saeb.filter(F.col("ano").isin(2019, 2021))
    .groupBy("ano", "rede")
    .count()
    .orderBy("ano", "rede")
)

Preenchimento das variáveis

In [0]:
display(
    df_saeb.filter(F.col("ano").isin(2019, 2021))
    .groupBy("ano")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.count("id_municipio").alias("com_municipio"),
        F.count("rede").alias("com_rede"),
        F.count("serie").alias("com_serie"),
        F.count("disciplina").alias("com_disciplina"),
        F.count("peso_aluno").alias("com_peso"),
        F.count("proficiencia").alias("com_proficiencia"),
        F.count("proficiencia_saeb").alias("com_proficiencia_saeb"),
    )
    .orderBy("ano")
)

## 3.3 Teste de Aderência

### Carregando SAEB

In [0]:
df_saeb_selecionado = (
    df_saeb.filter(F.col("ano") == 2019)
    .filter(F.col("serie") == 2)
    .filter(F.col("rede").isin("2", "3"))
    .filter(F.col("disciplina") == "LP")
)

In [0]:
display(
    df_saeb_selecionado.groupBy("rede").agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
        F.count("peso_aluno").alias("com_peso"),
        F.count("proficiencia_saeb").alias("com_proficiencia"),
    )
)

### Análise Completude

In [0]:
df_saeb_features = (
    df_saeb_selecionado.filter(
        F.col("peso_aluno").isNotNull() & F.col("proficiencia_saeb").isNotNull()
    )
    .groupBy("id_municipio", "rede")
    .agg(
        (
            F.sum(F.col("proficiencia_saeb") * F.col("peso_aluno"))
            / F.sum("peso_aluno")
        ).alias("saeb_lp_2ano_2019"),
        F.countDistinct("id_aluno").alias("saeb_qtd_alunos_2019"),
    )
    .withColumn("rede_codigo", F.col("rede").cast("string"))
)

df_cobertura_saeb = (
    df_base_atual.alias("base")
    .join(
        df_saeb_features.alias("saeb"),
        (F.col("base.id_municipio") == F.col("saeb.id_municipio"))
        & (F.col("base.rede") == F.col("saeb.rede_codigo")),
        how="left",
    )
    .select(
        "base.ano",
        "base.rede",
        F.col("saeb.saeb_lp_2ano_2019").alias("saeb_lp_2ano_2019"),
    )
)

In [0]:
display(
    df_cobertura_saeb.groupBy("ano", "rede")
    .agg(
        F.count("*").alias("qtd_alunos"), F.count("saeb_lp_2ano_2019").alias("com_saeb")
    )
    .withColumn("pct_saeb", F.round(F.col("com_saeb") / F.col("qtd_alunos") * 100, 2))
    .orderBy("ano", "rede")
)

### Testes

In [0]:
df_base_atual.select(
    "id_municipio", "rede"
).printSchema()

df_saeb_features.select(
    "id_municipio", "rede", "rede_codigo"
).printSchema()

In [0]:
display(
    df_base_atual
    .select("id_municipio", "rede")
    .distinct()
    .limit(10)
)

display(
    df_saeb_features
    .select("id_municipio", "rede", "rede_codigo")
    .distinct()
    .limit(10)
)

## 3.4 Conclusão

A utilização dos microdados do SAEB foi avaliada como possível fonte adicional de desempenho educacional histórico.

Entretanto, os identificadores de município presentes nas edições analisadas são códigos fictícios utilizados pelo Inep para proteção dos dados pessoais, não correspondendo aos códigos oficiais de município do IBGE.

Dessa forma, não é possível realizar uma associação confiável com a base analítica por município e rede.

# 4. Persistência das Features Selecionadas

Após as análises de cobertura, poder discriminatório e redundância, foram selecionadas duas métricas complementares do IDEB 2021:

- nota SAEB de Língua Portuguesa, representando desempenho educacional histórico;
- indicador de rendimento, representando fluxo escolar.

A tabela resultante possui granularidade município + rede e será utilizada na construção da segunda versão da base analítica.

In [0]:
df_ideb_features = df_ideb_selecionado.select(
    "id_municipio",
    "rede_codigo",
    F.col("nota_saeb_lingua_portuguesa").alias("ideb_saeb_portugues_2021"),
    F.col("indicador_rendimento").alias("ideb_indicador_rendimento_2021"),
)

In [0]:
(
    df_ideb_features.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.ml_features_ideb_2021")
)

In [0]:
df_ideb_features_salvo = spark.table(
    "workspace.gold.ml_features_ideb_2021"
)

display(
    df_ideb_features_salvo
    .agg(
        F.count("*").alias("qtd_contextos"),
        F.countDistinct(
            F.struct(
                "id_municipio",
                "rede_codigo"
            )
        ).alias("qtd_chaves_unicas"),
        F.count("ideb_saeb_portugues_2021")
        .alias("com_saeb_portugues"),
        F.count("ideb_indicador_rendimento_2021")
        .alias("com_indicador_rendimento")
    )
)

# 5. Conclusão

A segunda iteração de enriquecimento identificou o IDEB histórico como uma fonte relevante para a modelagem.

As métricas de desempenho educacional apresentaram poder discriminatório superior às features utilizadas na primeira versão da base, com ROC-AUC univariado próximo de 0,61 na validação.

Devido à elevada correlação entre as métricas do IDEB, foram selecionadas duas dimensões complementares:

- `ideb_saeb_portugues_2021`;
- `ideb_indicador_rendimento_2021`.

As features apresentam cobertura superior a 95% da população de modelagem e foram persistidas em `workspace.gold.ml_features_ideb_2021`.

Os microdados históricos do SAEB também foram avaliados, porém não foram incorporados devido à utilização de identificadores municipais mascarados, que impedem associação confiável com a base analítica.

A próxima etapa construirá a versão V2 da base de modelagem e avaliará o impacto das novas features utilizando os mesmos splits, modelos e parâmetros da primeira rodada.